# 1. Setup & Environment
In this cell, we install `datasets` and `pandas` to load and analyze the Amazon MASSIVE dataset.

In [21]:
!pip install -q datasets pandas

# 2. Download and Load the MASSIVE Dataset
In this cell, we load the `amazon/massive` dataset directly from Hugging Face using the `datasets` library.

In [30]:
import os
import tarfile
import glob
import urllib.request
import pandas as pd

# Official Amazon S3 link for MASSIVE 1.0
url = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz"
archive_path = "amazon-massive-dataset-1.0.tar.gz"

# Download archive if not present
if not os.path.exists(archive_path):
    print("Downloading MASSIVE dataset directly from Amazon S3...")
    urllib.request.urlretrieve(url, archive_path)
    print("Download complete!")

# Extract archive
print("Extracting files...")
with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path="massive_raw")

# Find all JSONL files
jsonl_files = glob.glob("massive_raw/**/*.jsonl", recursive=True)

# Load into Pandas DataFrame
df_list = []
for f in jsonl_files:
    df_list.append(pd.read_json(f, lines=True))

df_massive = pd.concat(df_list, ignore_index=True)

print(f"\nDataset successfully loaded! Total rows: {len(df_massive):,}")
print("\nSample Data Preview:")
df_massive.head(2)

Download complete!
Extracting files...


/tmp/ipykernel_923/1012239095.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="massive_raw")



Dataset successfully loaded! Total rows: 842,571

Sample Data Preview:


,id,locale,partition,scenario,intent,utt,annot_utt,worker_id,slot_method,judgments
0,0,sl-SL,test,alarm,alarm_set,zbudi me ob petih zjutraj ta teden,zbudi me ob [time : petih zjutraj] [date : ta ...,14,"[{'slot': 'time', 'method': 'translation'}, {'...","[{'worker_id': '0', 'intent_score': 1, 'slots_..."
1,1,sl-SL,train,alarm,alarm_set,zbudi me ob devetih zjutraj v petek,zbudi me ob [time : devetih zjutraj] v [date :...,14,"[{'slot': 'time', 'method': 'translation'}, {'...","[{'worker_id': '1', 'intent_score': 1, 'slots_..."


# 3. Identify Indian Languages & Inspect Dataset Columns
In this cell, we filter the dataset to identify all Indian languages (by locale code like `hi-IN`, `ta-IN`, `te-IN`, etc.), inspect all available columns, and check their data types.

In [31]:
# Print all column names and data types
print("--- DATASET COLUMNS & DATA TYPES ---")
print(df_massive.dtypes)
print("\n" + "="*50 + "\n")

# Identify Indian languages available in MASSIVE (locales ending in -IN or known Indian locales)
indian_locales = [loc for loc in df_massive['locale'].unique() if loc.endswith('-IN') or loc in ['hi-IN', 'ta-IN', 'te-IN', 'kn-IN', 'ml-IN', 'mr-IN', 'bn-IN', 'gu-IN', 'pa-IN', 'ur-IN']]

print(f"Indian Languages Found in MASSIVE ({len(indian_locales)} total):")
for loc in sorted(indian_locales):
    print(f" - {loc}")

# Create a filtered DataFrame for Indian languages only
df_indian = df_massive[df_massive['locale'].isin(indian_locales)].copy()
print(f"\nTotal rows across Indian languages: {len(df_indian):,}")

--- DATASET COLUMNS & DATA TYPES ---
id              int64
locale         object
partition      object
scenario       object
intent         object
utt            object
annot_utt      object
worker_id       int64
slot_method    object
judgments      object
dtype: object


Indian Languages Found in MASSIVE (5 total):
 - hi-IN
 - kn-IN
 - ml-IN
 - ta-IN
 - te-IN

Total rows across Indian languages: 82,605


# 4. Total Example Counts & Split Ratios per Indian Language
In this cell, we analyze the dataset distribution across Indian languages, calculating total sample counts per language alongside their exact train, validation (dev), and test split ratios.

In [32]:
import pandas as pd

# Group by locale (language) and partition (split)
split_counts = df_indian.groupby(['locale', 'partition']).size().unstack(fill_value=0)

# Calculate total examples per Indian language
split_counts['Total'] = split_counts.sum(axis=1)

# Calculate percentage split ratios
split_ratios = split_counts.copy()
for col in ['train', 'dev', 'test']:
    if col in split_ratios.columns:
        split_ratios[f'{col}_%'] = (split_ratios[col] / split_ratios['Total'] * 100).round(2)

# Display formatted table
print("--- INDIAN LANGUAGES: EXAMPLE COUNTS & SPLIT RATIOS ---")
display(split_ratios)

--- INDIAN LANGUAGES: EXAMPLE COUNTS & SPLIT RATIOS ---


partition,dev,test,train,Total,train_%,dev_%,test_%
locale,,,,,,,
hi-IN,2033,2974,11514,16521,69.69,12.31,18.0
kn-IN,2033,2974,11514,16521,69.69,12.31,18.0
ml-IN,2033,2974,11514,16521,69.69,12.31,18.0
ta-IN,2033,2974,11514,16521,69.69,12.31,18.0
te-IN,2033,2974,11514,16521,69.69,12.31,18.0


# 5. Unique Intents & Per-Intent Example Counts
In this cell, we extract all unique user intents present in the MASSIVE dataset and count how many examples exist for each intent across the Indian language subsets.

In [33]:
import pandas as pd

# Extract total intent counts across Indian languages
intent_counts = df_indian['intent'].value_counts().reset_index()
intent_counts.columns = ['Intent Name', 'Total Examples']

# Calculate percentage share of each intent
intent_counts['Percentage (%)'] = (intent_counts['Total Examples'] / len(df_indian) * 100).round(2)

print(f"Total Unique Intents Found: {len(intent_counts)}")
print("\n--- ALL INTENTS & EXAMPLE COUNTS ---")
pd.set_option('display.max_rows', 100)
display(intent_counts)

Total Unique Intents Found: 60

--- ALL INTENTS & EXAMPLE COUNTS ---


,Intent Name,Total Examples,Percentage (%)
0,calendar_set,5750,6.96
1,play_music,4690,5.68
2,weather_query,4275,5.18
3,general_quirky,4145,5.02
4,calendar_query,3970,4.81
5,qa_factoid,3875,4.69
6,news_query,3545,4.29
7,email_query,3050,3.69
8,email_sendemail,2655,3.21
9,datetime_query,2510,3.04


# 6. Group & Classify Intents by High-Level Scenarios (Domains)
In this cell, we group similar intents under their top-level scenarios (domains such as `alarm`, `calendar`, `iot`, `music`, etc.) and summarize the distribution of intents and total examples across these intent clusters.

In [34]:
import pandas as pd

# Group intents by high-level scenario domain
grouped_scenarios = df_indian.groupby(['scenario', 'intent']).size().reset_index(name='Example Count')

# Aggregate to see total intents and total examples per scenario
scenario_summary = df_indian.groupby('scenario').agg(
    Unique_Intents_Count=('intent', 'nunique'),
    Total_Examples=('utt', 'count'),
    Intents_List=('intent', lambda x: sorted(list(set(x))))
).reset_index()

# Sort by Total Examples descending
scenario_summary = scenario_summary.sort_values(by='Total_Examples', ascending=False)

print("--- HIGH-LEVEL INTENT DOMAINS & CLUSTER SUMMARY ---")
pd.set_option('display.max_colwidth', None)
display(scenario_summary[['scenario', 'Unique_Intents_Count', 'Total_Examples', 'Intents_List']])

--- HIGH-LEVEL INTENT DOMAINS & CLUSTER SUMMARY ---


,scenario,Unique_Intents_Count,Total_Examples,Intents_List
2,calendar,3,11850,"[calendar_query, calendar_remove, calendar_set]"
11,play,5,10120,"[play_audiobook, play_game, play_music, play_podcasts, play_radio]"
12,qa,5,8425,"[qa_currency, qa_definition, qa_factoid, qa_maths, qa_stock]"
5,email,4,6905,"[email_addcontact, email_query, email_querycontact, email_sendemail]"
7,iot,9,5535,"[iot_cleaning, iot_coffee, iot_hue_lightchange, iot_hue_lightdim, iot_hue_lightoff, iot_hue_lighton, iot_hue_lightup, iot_wemo_off, iot_wemo_on]"
6,general,3,4815,"[general_greet, general_joke, general_quirky]"
17,weather,1,4275,[weather_query]
16,transport,4,4025,"[transport_query, transport_taxi, transport_ticket, transport_traffic]"
8,lists,3,3965,"[lists_createoradd, lists_query, lists_remove]"
10,news,1,3545,[news_query]


# 7. Advanced Intent-Specific Analysis (Utterance Examples, Intent Lengths & Cross-Language Consistency)
In this cell, we perform deep intent-level analysis:
1. Extract top native-language sample utterances for every single intent.
2. Analyze word count & character length distributions per intent to understand intent complexity.
3. Verify intent balance across all Indian languages to ensure zero missing intents in translation.

In [35]:
import pandas as pd

# 1. Sample Utterances per Intent across Indian Languages
sample_utterances = (
    df_indian.groupby(['intent', 'locale'])['utt']
    .first()
    .unstack()
    .head(10)
)

print("--- 1. SAMPLE NATIVE UTTERANCES FOR TOP INTENTS ---")
display(sample_utterances)

# 2. Utterance Complexity (Length & Word Count) per Intent
df_indian['word_count'] = df_indian['utt'].apply(lambda x: len(str(x).split()))
df_indian['char_count'] = df_indian['utt'].apply(lambda x: len(str(x)))

intent_complexity = (
    df_indian.groupby('intent')
    .agg(
        Avg_Word_Count=('word_count', 'mean'),
        Avg_Char_Count=('char_count', 'mean'),
        Min_Words=('word_count', 'min'),
        Max_Words=('word_count', 'max')
    )
    .round(2)
    .sort_values(by='Avg_Word_Count', ascending=False)
)

print("\n--- 2. INTENT COMPLEXITY & UTTERANCE LENGTH METRICS ---")
display(intent_complexity.head(15))

# 3. Intent Consistency Matrix (Check if all Indian languages cover the exact same intents)
intent_lang_matrix = pd.crosstab(df_indian['intent'], df_indian['locale'])

print("\n--- 3. CROSS-LANGUAGE INTENT COVERAGE CHECK ---")
print(f"Are all intents present equally across all Indian languages? {intent_lang_matrix.nunique().to_dict()}")
display(intent_lang_matrix.head(10))

--- 1. SAMPLE NATIVE UTTERANCES FOR TOP INTENTS ---


locale,hi-IN,kn-IN,ml-IN,ta-IN,te-IN
intent,,,,,
alarm_query,मैंने कौन से अलार्म सेट किए हैं,ನನ್ನ ಯಾವ ಅಲಾರಾಂಗಳನ್ನು ಹೊಂದಿಸಿದ್ದೇನೆ,ഞാൻ എന്ത് അലാറമാണ് സജ്ജീകരിച്ചിരിക്കുന്നത്,நான் என்ன அலாரங்களை அமைத்துள்ளேன்,నేను ఏమి అలారంలు పెట్టాను
alarm_remove,मेरा सुबह सात बजे का अलार्म रद्द करो,ನನ್ನ ಮುಂಜಾನೆ ಏಳು ಗಂಟೆಗೆ ಮುಂಜಾನೆ ಅಲಾರಂ ಅನ್ನು ರದ್ದುಮಾಡಿ,എൻ്റെ രാവിലെ ഏഴിന് ഉള്ള അലാറം റദ്ദാക്കൂ,எனது காலை ஏழு மணி அலாரத்தை ரத்து செய்,నా ఏడు ఏ.ఎం. అలారం రద్దు చేయి
alarm_set,मुझे इस सप्ताह सुबह पांच बजे जगा दो,ಈ ವಾರ ಬೆಳಗ್ಗೆ ಐದು ಗಂಟೆಗೆ ನನ್ನನ್ನು ಎದ್ದೇಳಿ,ഈ ആഴ്ച പുലർച്ചെ അഞ്ച് മണിക്ക് എന്നെ ഉണർത്തുക,இந்த வாரம் காலை ஐந்து மணிக்கு என்னை எழுப்பு,ఈ వారం ఉదయం ఐదు గంటలకు నన్ను మేల్కొలపండి
audio_volume_down,आवाज़ को कम करो,ವಾಲ್ಯೂಮ್ ಅನ್ನು ಕಡಿಮೆಗೆ ಹೊಂದಿಸಿ,ശബ്ദം ഏറ്റവും താഴത്തേയ്ക്ക് ആക്കുക,சத்தத்தை குறைவாக வைக்கவும்,వాల్యూమ్‌ను తక్కువగా సెట్ చేయండి
audio_volume_mute,बंद करो,ಸ್ತಬ್ಧ,ശബ്ദം കുറക്കുക,அமைதியான,నిశ్శబ్దంగా
audio_volume_other,वॉल्यूम बदलें,ವಾಲ್ಯೂಮ್ ಅನ್ನು ಬದಲಾಯಿಸಿ,ഒച്ച മാറ്റുക,ஒலியின் அளவை மாற்றவும்,వాల్యూమ్ మార్చు
audio_volume_up,वॉल्यूम बढ़ाओ,ವಾಲ್ಯೂಮ್ ಅನ್ನು ಹೆಚ್ಚಿಸಿ,ഒച്ച കൂട്ടുക,ஒலி அளவை உயர்த்தவும்,వాల్యూమ్ పెంచండి
calendar_query,जांचें कि show कब शुरू होता है,ಪ್ರದರ್ಶನ ಯಾವಾಗ ಪ್ರಾರಂಭವಾಗುತ್ತದೆ ಎಂಬುದನ್ನು ಪರಿಶೀಲಿಸಿ,ഷോ എപ്പോൾ തുടങ്ങുമെന്ന് പരിശോധിക്കുക,நிகழ்ச்சி எப்போது தொடங்குகிறது என்பதை சரிபார்க்கவும்,ప్రదర్శన ఎప్పుడు ప్రారంభమవుతుందో తనిఖీ చేయండి
calendar_remove,बुधवार को व्यावसायिक बैठक रद्द करें,ಬುಧವಾರದ ವ್ಯವಹಾರ ಸಭೆಯನ್ನು ರದ್ದುಗೊಳಿಸಿ,ബുധനാഴ്ചത്തെ ബിസിനസ് മീറ്റിംഗ് റദ്ദാക്കുക,புதன்கிழமை வணிக கூட்டத்தை ரத்து செய்யுங்கள்,బుధవారం వ్యాపార సమావేశాన్ని రద్దు చేయండి



--- 2. INTENT COMPLEXITY & UTTERANCE LENGTH METRICS ---


,Avg_Word_Count,Avg_Char_Count,Min_Words,Max_Words
intent,,,,
datetime_convert,8.21,51.43,1,21
calendar_set,8.11,57.30,1,32
social_post,7.74,54.13,1,29
transport_ticket,7.71,54.06,2,24
cooking_query,7.63,48.23,1,18
email_sendemail,7.37,49.56,1,35
takeaway_order,7.35,48.39,1,20
email_addcontact,7.09,48.19,2,18
transport_query,6.59,46.37,1,22



--- 3. CROSS-LANGUAGE INTENT COVERAGE CHECK ---
Are all intents present equally across all Indian languages? {'hi-IN': 55, 'kn-IN': 55, 'ml-IN': 55, 'ta-IN': 55, 'te-IN': 55}


locale,hi-IN,kn-IN,ml-IN,ta-IN,te-IN
intent,,,,,
alarm_query,183,183,183,183,183
alarm_remove,113,113,113,113,113
alarm_set,254,254,254,254,254
audio_volume_down,71,71,71,71,71
audio_volume_mute,157,157,157,157,157
audio_volume_other,24,24,24,24,24
audio_volume_up,135,135,135,135,135
calendar_query,794,794,794,794,794
calendar_remove,426,426,426,426,426


# 8. Slot & Entity Annotation Mapping per Intent
In this cell, we parse the `annot_utt` column to extract all slot/entity labels (e.g., `[time : ...]`, `[date : ...]`) and map which entities co-occur with each intent across the Indian language utterances.

In [36]:
import re
import pandas as pd
from collections import Counter

# Function to extract slot/entity names from annotated utterances
def extract_slots(annot_str):
    if not isinstance(annot_str, str):
        return []
    # Matches patterns like [slot_name : slot_value]
    return re.findall(r'\[\s*([a-zA-Z_]+)\s*:', annot_str)

# Extract slots for every sample
df_indian['extracted_slots'] = df_indian['annot_utt'].apply(extract_slots)

# Explode dataset to analyze slot-intent co-occurrence
slots_exploded = df_indian.explode('extracted_slots')
slots_exploded = slots_exploded.dropna(subset=['extracted_slots'])

# Summarize top slots per intent
intent_slot_summary = (
    slots_exploded.groupby(['scenario', 'intent'])['extracted_slots']
    .agg(
        Total_Slots_Used='count',
        Unique_Slots_Count='nunique',
        Associated_Entities=lambda x: [item for item, _ in Counter(x).most_common(10)]
    )
    .reset_index()
    .sort_values(by='Total_Slots_Used', ascending=False)
)

print("--- TOP INTENTS AND THEIR ASSOCIATED ENTITIES / SLOTS ---")
pd.set_option('display.max_colwidth', None)
display(intent_slot_summary)

--- TOP INTENTS AND THEIR ASSOCIATED ENTITIES / SLOTS ---


,scenario,intent,Total_Slots_Used,Unique_Slots_Count,Associated_Entities
9,calendar,calendar_set,11558,17,"[event_name, date, time, person, general_frequency, relation, place_name, timeofday, meal_type, media_type]"
59,weather,weather_query,5726,10,"[weather_descriptor, date, place_name, timeofday, time, event_name, general_frequency, meal_type, business_type, food_type]"
7,calendar,calendar_query,4673,12,"[date, event_name, time, timeofday, person, relation, place_name, sport_type, meal_type, business_name]"
40,play,play_music,4398,14,"[artist_name, music_genre, song_name, playlist_name, music_descriptor, player_setting, date, media_type, app_name, time]"
37,news,news_query,3351,10,"[news_topic, media_type, place_name, date, person, time, general_frequency, timeofday, transport_type, device_type]"
51,social,social_post,2723,11,"[media_type, business_name, event_name, person, relation, date, place_name, personal_info, device_type, weather_descriptor]"
17,email,email_sendemail,2705,14,"[person, relation, date, event_name, email_address, time, timeofday, meal_type, place_name, general_frequency]"
45,qa,qa_factoid,2669,11,"[person, place_name, artist_name, event_name, date, time, news_topic, food_type, list_name, movie_name]"
55,transport,transport_query,2531,14,"[place_name, transport_type, date, time, business_name, transport_name, timeofday, business_type, event_name, relation]"
13,datetime,datetime_query,2126,7,"[date, place_name, time_zone, event_name, time, food_type, timeofday]"


# 9. Intent Ambiguity & Overlap Analysis
In this cell, we detect intent overlap and potential classification ambiguity across Indian languages:
1. Find duplicate utterances assigned to different intents (annotation conflicts).
2. Identify intents sharing identical entity structures (high-confusion intent pairs).

In [37]:
import pandas as pd

# 1. Detect Identical Utterances Mapped to Multiple Intents (Ambiguous/Conflicting Intents)
ambiguous_utts = (
    df_indian.groupby(['locale', 'utt'])['intent']
    .nunique()
    .reset_index(name='intent_count')
)
ambiguous_utts = ambiguous_utts[ambiguous_utts['intent_count'] > 1]

print(f"--- 1. AMBIGUOUS UTTERANCES (Same text, different intents) ---")
print(f"Total conflicting utterance instances across Indian languages: {len(ambiguous_utts)}")

if len(ambiguous_utts) > 0:
    conflicting_samples = df_indian.merge(
        ambiguous_utts[['locale', 'utt']], on=['locale', 'utt']
    )[['locale', 'utt', 'intent']].drop_duplicates()
    display(conflicting_samples.head(10))
else:
    print("No direct text collisions found across intents!")

# 2. Intent Confusion Pairs (Intents sharing identical slot signatures)
df_indian['slot_signature'] = df_indian['extracted_slots'].apply(lambda s: tuple(sorted(s)))

intent_signature_sharing = (
    df_indian.groupby(['scenario', 'intent'])['slot_signature']
    .agg(lambda x: tuple(set(x)))
    .reset_index()
)

print("\n--- 2. INTENT SLOT SIGNATURE PROFILE ---")
display(intent_signature_sharing.head(10))

--- 1. AMBIGUOUS UTTERANCES (Same text, different intents) ---
Total conflicting utterance instances across Indian languages: 113


,locale,utt,intent
0,te-IN,నిశ్శబ్దంగా,audio_volume_mute
1,te-IN,దయచేసి లైట్లను ఆఫ్ చేయండి,iot_hue_lightoff
2,te-IN,మీరు ఎలా ఉన్నారు,general_greet
3,te-IN,నిశ్శబ్దంగా,audio_volume_down
4,te-IN,లైట్లు ఆఫ్ చేయండి,iot_hue_lightoff
5,te-IN,లైట్స్ మార్చు,iot_hue_lightoff
6,te-IN,లైట్లు ఆర్పివేయండి,iot_hue_lightup
7,te-IN,దయచేసి లైట్లను ఆఫ్ చేయండి,iot_hue_lightdim
8,te-IN,సాకెట్ ఆన్ చేయండి,iot_wemo_on
9,te-IN,శుభ రాత్రి,audio_volume_mute



--- 2. INTENT SLOT SIGNATURE PROFILE ---


,scenario,intent,slot_signature
0,alarm,alarm_query,"((date, event_name), (date, timeofday), (alarm_type, date), (time,), (timeofday,), (event_name,), (alarm_type, time), (event_name, event_name), (date, time), (house_place,), (date,), (event_name, timeofday), (alarm_type,), (time, timeofday), (), (device_type,), (alarm_type, timeofday))"
1,alarm,alarm_remove,"((date, event_name), (date, timeofday), (relation,), (time, time), (alarm_type, date), (), (timeofday,), (date, time), (date,), (alarm_type,), (time,), (person,))"
2,alarm,alarm_set,"((date, time, timeofday), (date, timeofday), (timeofday,), (date, date, time), (date, event_name, timeofday), (date,), (event_name, time), (time, time), (date, date, time, timeofday), (date, time, time, timeofday), (event_name, time, timeofday), (date, event_name), (time_zone,), (general_frequency, time), (date, event_name, event_name, time), (event_name,), (date, order_type, time), (date, time, time), (date, time), (date, media_type), (time, timeofday), (time,), (event_name, relation, timeofday), (date, event_name, time), (date, event_name, person, time), (date, event_name, time, timeofday), (time, timeofday, timeofday), (date, event_name, relation, time), (alarm_type, date, time, timeofday), (alarm_type, time), (alarm_type, date, time), ())"
3,audio,audio_volume_down,"((change_amount,), (), (device_type,))"
4,audio,audio_volume_mute,"((device_type, time), (event_name,), (timeofday,), (), (time, time, timeofday), (date,), (change_amount,), (time,), (device_type,))"
5,audio,audio_volume_other,"((change_amount,), ())"
6,audio,audio_volume_up,"((change_amount, device_type), (change_amount,), (song_name,), (), (device_type,), (media_type,))"
7,calendar,calendar_query,"((date, time, timeofday), (date, timeofday), (list_name,), (meal_type, timeofday), (date, relation, time), (event_name, person), (timeofday,), (event_name, time, time), (event_name, relation), (date, date), (date, person, time), (sport_type, timeofday), (date, event_name, timeofday), (event_name, event_name), (event_name, general_frequency), (date,), (date, date, event_name), (date, event_name, event_name, meal_type), (event_name, person, relation), (event_name, place_name), (event_name, time), (time, time), (date, meal_type), (date, relation, timeofday), (date, time, time, timeofday), (date, person), (date, event_name, time, time), (date, event_name, meal_type), (event_name, person, time), (date, relation), (person,), (date, sport_type), (event_name, time, timeofday), (date, event_name), (date, event_name, relation), (date, event_name, person), (relation,), (sport_type, sport_type), (sport_type,), (business_name,), (event_name,), (date, event_name, event_name), (business_name, event_name), (date, time, time), (date, time), (person, time), (event_name, timeofday), (time, timeofday), (date, place_name), (person, place_name), (time,), (date, event_name, person, time, time), (place_name,), (date, event_name, place_name), (date, event_name, time), (date, event_name, person, time), (date, event_name, place_name, time), (event_name, person, timeofday), (date, list_name), (), (date, place_name, time))"
8,calendar,calendar_remove,"((list_name,), (date, timeofday), (meal_type, timeofday), (event_name, person), (date, date), (event_name, event_name), (app_name, event_name), (date, event_name, timeofday), (event_name, general_frequency), (date,), (event_name, event_name, place_name), (business_type, date, meal_type), (event_name, place_name), (event_name, time), (date, meal_type), (event_name, general_frequency, time), (meal_type, person), (person,), (meal_type, relation), (date, event_name), (date, event_name, relation), (date, event_name, person), (event_name,), (date, event_name, event_name), (date, time), (person, time), (meal_type, place_name), (time,), (place_name, time, transport_type), (date, event_name, time), (date, meal_type, relation), (event_name, person, timeofday), (), (date, meal_type, time))"
9,calendar,calendar_set,"((event

# 10. Export Indian Intent Analysis & Summaries to CSV
In this cell, we save all the generated intent tables, domain summaries, and slot mappings into downloadable CSV files directly to your local computer.

In [38]:
import pandas as pd
from google.colab import files

# 1. Export Full Filtered Indian Languages Dataset
df_indian.to_csv("massive_indian_languages_full.csv", index=False)
print("Saved: massive_indian_languages_full.csv")

# 2. Export Split Ratios Summary Table
split_ratios.to_csv("indian_languages_split_ratios.csv")
print("Saved: indian_languages_split_ratios.csv")

# 3. Export All Intents & Example Counts
intent_counts.to_csv("intents_example_counts.csv", index=False)
print("Saved: intents_example_counts.csv")

# 4. Export Scenario & High-Level Domain Summary
scenario_summary.to_csv("scenario_domain_summary.csv", index=False)
print("Saved: scenario_domain_summary.csv")

# 5. Export Intent-Slot Mapping Table
intent_slot_summary.to_csv("intent_slot_mapping.csv", index=False)
print("Saved: intent_slot_mapping.csv")

print("\n--- INITIATING FILE DOWNLOADS ---")

# Trigger automatic download for Colab
files.download("massive_indian_languages_full.csv")
files.download("indian_languages_split_ratios.csv")
files.download("intents_example_counts.csv")
files.download("scenario_domain_summary.csv")
files.download("intent_slot_mapping.csv")

print("All analysis CSV files have been exported successfully!")

Saved: massive_indian_languages_full.csv
Saved: indian_languages_split_ratios.csv
Saved: intents_example_counts.csv
Saved: scenario_domain_summary.csv
Saved: intent_slot_mapping.csv

--- INITIATING FILE DOWNLOADS ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All analysis CSV files have been exported successfully!
